In [2]:
import numpy as np
import pandas as pd
import geopandas as gpd
from config import *

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
nlcdClasses = {
    11: "Open Water",
    12: "Perennial Ice/Snow",
    21: "Developed Open Space",
    22: "Developed Low Intensity",
    23: "Developed Medium Intensity",
    24: "Developed High Intensity",
    31: "Barren Land",
    41: "Deciduous Forest",
    42: "Evergreen Forest",
    43: "Mixed Forest",
    52: "Shrub/Scrub",
    71: "Grassland/Herbaceous",
    81: "Pasture/Hay",
    82: "Cultivated Crops",
    90: "Woody Wetlands",
    95: "Emergent Herbaceous Wetlands"}

In [9]:
LU_site_data = pd.read_csv(f"{land_use_filepath}/{site}.csv")
LU_site_data.head()

,DateTime,Open Water,Perennial Ice/Snow,Developed Open Space,Developed Low Intensity,Developed Medium Intensity,Developed High Intensity,Barren Land,Deciduous Forest,Evergreen Forest,Mixed Forest,Shrub/Scrub,Grassland/Herbaceous,Pasture/Hay,Cultivated Crops,Woody Wetlands,Emergent Herbaceous Wetlands
0,1985,0.002296,0,0.038428,0.027743,0.005827,0.001139,0.000396,0.118027,0.000101,0.001560,0.000143,0.000439,0.030715,0.756007,0.016214,0.000967
1,1986,0.002586,0,0.038600,0.027758,0.005831,0.001139,0.000406,0.118032,0.000101,0.001562,0.000144,0.000437,0.031866,0.754410,0.016198,0.000932
2,1987,0.002605,0,0.038903,0.027801,0.005858,0.001146,0.000409,0.118104,0.000103,0.001574,0.000142,0.000415,0.031265,0.754592,0.016174,0.000911
3,1988,0.002392,0,0.039232,0.027844,0.005862,0.001146,0.000413,0.118455,0.000106,0.001612,0.000139,0.000422,0.030436,0.754856,0.016169,0.000915
4,1989,0.002824,0,0.039309,0.028039,0.006004,0.001176,0.000415,0.118692,0.000106,0.001628,0.000148,0.000394,0.031063,0.753246,0.016095,0.000861


In [23]:
LU_data = []
ave_LU_site_data = []

# Aggregating some categories for future use
for site in MAIN_SITES:
    # Load the shapefile
    LU_site_data = pd.read_csv(f"{land_use_filepath}/{site}.csv")
    LU_site_data.drop(columns = 'Perennial Ice/Snow', inplace=True)
    LU_site_data['Developed Total'] = LU_site_data['Developed Open Space'] + LU_site_data['Developed Low Intensity'] + LU_site_data['Developed Medium Intensity'] + LU_site_data['Developed High Intensity']
    LU_site_data['Developed Built Total'] = LU_site_data['Developed Low Intensity'] + LU_site_data['Developed Medium Intensity'] + LU_site_data['Developed High Intensity']
    LU_site_data['Modified Total'] = LU_site_data['Cultivated Crops'] + LU_site_data['Developed Total']
    LU_site_data['Forested Total'] = LU_site_data['Deciduous Forest'] + LU_site_data['Evergreen Forest'] + LU_site_data['Mixed Forest']
    LU_site_data['Grass and pasture'] = LU_site_data['Grassland/Herbaceous'] + LU_site_data['Pasture/Hay']
    LU_site_data['Wetlands'] = LU_site_data['Woody Wetlands'] + LU_site_data['Emergent Herbaceous Wetlands']

    LU_site_data['STREAM_ID'] = site
    col = (LU_site_data.pop('STREAM_ID'))
    LU_site_data.insert(0, column='STREAM_ID', value=col)

    LU_data.append(LU_site_data)

    site_lu_data = LU_site_data[LU_site_data['DateTime'] >= START_YEAR].drop(columns=['DateTime','STREAM_ID']).mean().to_frame().T
    site_lu_data.insert(0, 'STREAM_ID', site)
    ave_LU_site_data.append(site_lu_data)

LU_data = pd.concat(LU_data, ignore_index=True)
averaged_LU_data = pd.concat(ave_LU_site_data, ignore_index=True)

In [27]:
LU_data.to_csv(OUTPUT_filepath+'temporal_LU_data.csv', index = False)
averaged_LU_data.to_csv(OUTPUT_filepath+'averaged_LU_data.csv', index = False)